### **Pass information into the notebook automatically after each file copy**
### 

In [1]:
# Pipeline parameters

batch_id = ""
table_name = ""
source_path = ""
load_status = "SUCCESS"
error_message = ""

StatementMeta(, 1da59fee-54da-4c81-baa5-dffa7ab1f178, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

# Audit table configuration

AUDIT_TABLE = "bronze_ingestion_audit"

AUDIT_SCHEMA = StructType([
    StructField("batch_id", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("source_path", StringType(), True),
    StructField("load_status", StringType(), False),
    StructField("load_timestamp", TimestampType(), True),
    StructField("error_message", StringType(), True)
])

# Create audit record

def log_ingestion() -> None:
    """Write one ingestion result to the Bronze audit table."""

    audit_data = [(
        batch_id,
        table_name,
        source_path,
        load_status,
        None,
        error_message if error_message else None
    )]

    audit_df = spark.createDataFrame(
        audit_data,
        AUDIT_SCHEMA
    )

    audit_df = audit_df.withColumn(
        "load_timestamp",
        F.current_timestamp()
    )

    (audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE))

    print(f"{table_name:<15} {load_status}")

log_ingestion()

StatementMeta(, 1da59fee-54da-4c81-baa5-dffa7ab1f178, 4, Finished, Available, Finished, True)

                SUCCESS
